# E012 segmented-budget counterfactual reproduction

本 notebook 只读取 GitHub 归档的 compact JSON，不读取 NPZ、RGB、原始 trajectory 或 GPU 运行目录。权威计算仍由 `scripts/analyze_e012_budget_counterfactual.py` 提供。

In [ ]:
from pathlib import Path
import importlib.util
import json

repo_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'pyproject.toml').is_file()),
    None,
)
assert repo_root is not None, '请从 robot-vla 仓库内运行 notebook'
script_path = repo_root / 'scripts' / 'analyze_e012_budget_counterfactual.py'
spec = importlib.util.spec_from_file_location('e012_budget_analysis', script_path)
assert spec is not None and spec.loader is not None
analysis_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(analysis_module)


In [ ]:
results_root = repo_root / 'docs' / 'results' / 'e012'
analysis = analysis_module.build_analysis(
    collection_summary_path=results_root / 'collection_summary.json',
    smoke_root=results_root / 'segmented-budget-smoke',
    counterfactual_root=results_root / 'segmented-budget-counterfactual',
)
observed = analysis['observed_counterfactual']
planning = analysis['capacity_planning']
counts = observed['classification_counts']


In [ ]:
assert observed['count'] == 16
assert counts == {
    'recovered_full_eligible': 5,
    'expert_completed_but_snapshot_or_paired_gate_failed': 1,
    'expert_recovery_budget_exhausted': 4,
    'other_behavioral_rejection': 6,
    'prefix_mismatch': 0,
    'engineering_error': 0,
}
assert observed['hard_deadline_count'] == 0
assert planning['planning_point_rate'] == 0.15
assert planning['gate_required_eligible'] == 20
assert planning['minimum_pool_sizes_by_model'][-1]['beta_binomial_jeffreys'] == 223
assert analysis['trajectory_usage'] == 'forbidden as training data'


In [ ]:
summary = {
    'classification_counts': counts,
    'eligible_expert_actions': observed['recovered_expert_actions'],
    'planning_rate': planning['planning_point_rate'],
    'pool_options': planning['pool_options'],
    'jeffreys_95_minimum_pool': planning['minimum_pool_sizes_by_model'][-1]['beta_binomial_jeffreys'],
}
print(json.dumps(summary, ensure_ascii=False, indent=2))
